# BBC News Analysis with LangChain + Ollama (SLMs)

This notebook analyzes BBC news articles using **LangChain** with **local small language models (SLMs) via Ollama**, so we can process the **entire dataset (2,225 articles)** without hitting API rate limits.

For each article we:
1. **Topic Classification** – detect the main topic
2. **Categorize** – Business, Entertainment, Politics, Sport, or Tech
3. **Summarization** – a short 2–3 sentence summary
4. **Key Entity Extraction** – important people, organizations, places

**Balancing speed vs accuracy:**
- Classification is an *easy* task with a one-word output -> we use a **fast, tiny model** (`llama3.2:1b`).
- Summarization & entity extraction need more capability -> we use a **stronger small model** (`llama3.2:3b`).
- For the full-dataset run we use **one combined call per article** (topic + summary + entities together) instead of three separate calls -> ~3x fewer calls.

## Step 0: Set up Ollama (one-time)

1. Install Ollama from https://ollama.com/download and make sure the app / server is running.
2. Pull the models we use (run these in a terminal):

```bash
ollama pull llama3.2:1b
ollama pull llama3.2:3b
```

3. Install the Python packages (uncomment below).

can swap in any SLM, e.g. `qwen2.5:3b`, `gemma2:2b`, `phi3.5`, or `mistral`.

In [ ]:
# !pip install langchain langchain-ollama pandas

In [1]:
import os, json, re, time, ast
import pandas as pd

### Initialize the LLMs (ChatModels) via LangChain + Ollama

`ChatOllama` runs a local model. Two useful speed/accuracy knobs:
- `temperature=0` -> deterministic, consistent outputs
- `num_predict` -> caps output length, which makes generation faster (classification only needs a word, so we cap it very low)

In [29]:
from langchain_ollama import ChatOllama

FAST_MODEL    = "llama3.2:1b"   # very fast; fine for simple category labels
QUALITY_MODEL = "llama3.2:3b"   # stronger; better summaries & entity extraction

# One model per use case (speed vs accuracy trade-off):
classify_llm  = ChatOllama(model=FAST_MODEL,    temperature=0, num_predict=8)    # tiny output -> fast
summarize_llm = ChatOllama(model=QUALITY_MODEL, temperature=0, num_predict=200)  # needs quality
entity_llm    = ChatOllama(model=QUALITY_MODEL, temperature=0, num_predict=200)  # needs quality

# quick sanity check (confirms Ollama is running and the model is pulled)
print(classify_llm.invoke("Reply with just the word: ready").content)

No.


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Step 1: Load the Dataset

The BBC News dataset is **tab-separated** with columns: `category`, `filename`, `title`, `content`.

Set `NUM_ARTICLES`:
- `None` -> process **ALL 2,225 rows** (needed for the bonus).
- a number (e.g. `30`) -> quick test run first.

In [30]:
# The file is TAB separated, so we pass sep='\t'
df_full = pd.read_csv("bbc-news-data.csv", sep="\t")
print("Full dataset shape:", df_full.shape)
print("Categories:", list(df_full["category"].unique()))
df_full.head()

Full dataset shape: (2225, 4)
Categories: ['business', 'entertainment', 'politics', 'sport', 'tech']


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [31]:
NUM_ARTICLES = None   # None = ALL rows, Set to 30 to test first.

df = (df_full if NUM_ARTICLES is None else df_full.head(NUM_ARTICLES)).copy().reset_index(drop=True)

# Give each article a simple ID and tidy column names
df["Article_ID"] = df.index + 1
df = df.rename(columns={"title": "Title", "content": "Article_Text", "category": "Original_Category"})
df = df[["Article_ID", "Title", "Article_Text", "Original_Category"]]

print("Working dataset shape:", df.shape)
df.head()

Working dataset shape: (2225, 4)


,Article_ID,Title,Article_Text,Original_Category
0,1,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...,business
1,2,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...,business
2,3,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...,business
3,4,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,business
4,5,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...,business


## Step 2: Topic Classification Task

A LangChain chain (`prompt | llm | parser`) that returns a **single category label**, using a few-shot prompt
and the **fast** model.

In [32]:
classify_system = """You are a news topic classifier.
Read the news article and classify it into EXACTLY ONE of these categories:
Business, Entertainment, Politics, Sport, Tech.

Rules:
- Reply with ONLY the single category word. No explanation, no punctuation.

Examples:
Article: "Shares rose 5% after the company reported record quarterly profits and strong sales."
Category: Business

Article: "The blockbuster film topped the box office this weekend to rave reviews from critics."
Category: Entertainment

Article: "The prime minister faced questions in parliament over the new tax bill and election."
Category: Politics

Article: "The striker scored a hat-trick as his team won the championship final at the stadium."
Category: Sport

Article: "The firm unveiled a new smartphone chip that doubles processing speed and battery life."
Category: Tech
"""

classify_prompt = ChatPromptTemplate([
    ("system", classify_system),
    ("human", "Article:\n{article}\n\nCategory:"),
])

classify_chain = classify_prompt | classify_llm | StrOutputParser()

In [33]:
VALID_TOPICS = ["Business", "Entertainment", "Politics", "Sport", "Tech"]

def clean_topic(raw_text):
    text = raw_text.strip()
    for topic in VALID_TOPICS:
        if topic.lower() in text.lower():
            return topic
    return text  # fallback: whatever the model returned

def classify_article(article_text):
    return clean_topic(classify_chain.invoke({"article": article_text}))

**Expected Output: works on a sample datapoint.**

In [36]:
sample_text = df["Article_Text"].iloc[0]
print("TITLE:", df["Title"].iloc[0])
print("TRUE CATEGORY:", df["Original_Category"].iloc[0])
print("PREDICTED TOPIC:", classify_article(sample_text))

TITLE: Ad sales boost Time Warner profit
TRUE CATEGORY: business
PREDICTED TOPIC: Business


## Step 3: Summarization Task

A chain that summarizes each article in **2–3 sentences** using the **quality** model.

In [37]:
summarize_system = """You are a news summarizer.
Summarize the main points of the news article in 2 to 3 clear sentences.
Capture who, what, when, where, and why where applicable.
Do not add personal opinions or commentary. Return only the summary text."""

summarize_prompt = ChatPromptTemplate([
    ("system", summarize_system),
    ("human", "News article:\n{article}\n\nSummary:"),
])

summarize_chain = summarize_prompt | summarize_llm | StrOutputParser()

def summarize_article(article_text):
    return summarize_chain.invoke({"article": article_text}).strip()

**Expected Output: works on a sample datapoint.**

In [38]:
print("SUMMARY:\n", summarize_article(sample_text))

SUMMARY:
 Time Warner's quarterly profits increased 76% to $1.13 billion, driven by sales of high-speed internet connections and higher advert sales. The company's fourth-quarter sales rose 2% to $11.1 billion, with its internet business AOL experiencing a 8% increase in underlying profit. Time Warner now owns 8% of Google and is projecting 5% operating earnings growth for 2005.


## Step 4: Key Entity Extraction

A chain that lists key **people, organizations, and places** as a JSON list, using the **quality** model.
A robust parser falls back to comma-splitting if the small model returns imperfect JSON.

In [40]:
def parse_entities(raw):
    """Turn a model response into a clean Python list of entities."""
    try:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        data = json.loads(match.group(0)) if match else json.loads(raw)
        ents = data.get("entities", [])
        if isinstance(ents, list):
            return [str(e).strip() for e in ents if str(e).strip()]
    except Exception:
        pass
    cleaned = re.sub(r"[\[\]{}\"]", "", raw)
    return [p.strip() for p in cleaned.split(",") if p.strip()][:10]

In [41]:
entity_system = """You extract key named entities from a news article.
Identify important PEOPLE, ORGANIZATIONS, and PLACES mentioned.

Return ONLY a JSON object in exactly this format (no extra text):
{{"entities": ["entity one", "entity two", "entity three"]}}"""

entity_prompt = ChatPromptTemplate([
    ("system", entity_system),
    ("human", "Article:\n{article}\n\nJSON:"),
])

entity_chain = entity_prompt | entity_llm | StrOutputParser()

def extract_entities(article_text):
    return parse_entities(entity_chain.invoke({"article": article_text}).strip())

**Expected Output: works on a sample datapoint.**

In [42]:
print("KEY ENTITIES:", extract_entities(sample_text))

KEY ENTITIES: ['TimeWarner', 'Google', 'Warner Bros', 'AOL', 'US Securities Exchange Commission', 'Richard Parsons', 'Lord of the Rings', 'Bertelsmann']


## Step 5: Process the whole dataset (efficiently) and update the DataFrame

Running three separate calls per article over 2,225 rows = ~6,675 calls. To make the full run practical,
we combine all three tasks into **one call per article** that returns a single JSON object with
`topic`, `summary`, and `entities`. This is the main speed optimization for the bonus.

We use the **quality** model for the combined call, cap the output length, and add **progress + ETA**
plus **checkpoint saving** so a long run is safe.

In [43]:
# One combined chain that does topic + summary + entities in a single call
analyze_llm = ChatOllama(model=QUALITY_MODEL, temperature=0, num_predict=320)

analyze_system = """You are a news analysis assistant.
Analyze the news article and return ONLY a JSON object with EXACTLY these keys:
- "topic": exactly one of Business, Entertainment, Politics, Sport, Tech
- "summary": a factual 2-3 sentence summary, no opinions
- "entities": a JSON list of important people, organizations, and places

Return only valid JSON and nothing else. Example:
{{"topic": "Business", "summary": "...", "entities": ["Name A", "Org B", "Place C"]}}"""

analyze_prompt = ChatPromptTemplate([
    ("system", analyze_system),
    ("human", "Article:\n{article}\n\nJSON:"),
])

analyze_chain = analyze_prompt | analyze_llm | StrOutputParser()

def analyze_article(article_text):
    """Return (topic, summary, entities) from a single combined LLM call."""
    raw = analyze_chain.invoke({"article": article_text}).strip()
    topic, summary, entities = "Unknown", "", []
    try:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        data = json.loads(match.group(0)) if match else json.loads(raw)
        topic = clean_topic(str(data.get("topic", "Unknown")))
        summary = str(data.get("summary", "")).strip()
        ents = data.get("entities", [])
        entities = [str(e).strip() for e in ents if str(e).strip()] if isinstance(ents, list) else []
    except Exception:
        # if JSON failed, fall back to the three separate chains for this one article
        topic = classify_article(article_text)
        summary = summarize_article(article_text)
        entities = extract_entities(article_text)
    return topic, summary, entities

### Run the analysis over every row
Progress prints every 10 rows with elapsed time and an ETA; results are checkpointed to CSV every 100 rows.

In [44]:
OUTPUT_CSV       = "bbc_news_analysis_full.csv"
PRINT_EVERY      = 10
CHECKPOINT_EVERY = 100

topics, summaries, entities_list = [], [], []
total = len(df)
start = time.time()

for i, row in df.iterrows():
    try:
        topic, summary, entities = analyze_article(row["Article_Text"])
    except Exception as e:
        print(f"Row {i+1} error: {e}")
        topic, summary, entities = "ERROR", "ERROR", []

    topics.append(topic)
    summaries.append(summary)
    entities_list.append(entities)

    done = i + 1
    if done % PRINT_EVERY == 0 or done == total:
        elapsed = time.time() - start
        rate = done / elapsed
        eta = (total - done) / rate if rate > 0 else 0
        print(f"[{done}/{total}] {rate:.2f} art/s | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min")

    # checkpoint: save progress so a long run isn't lost
    if done % CHECKPOINT_EVERY == 0:
        ckpt = df.iloc[:done].copy()
        ckpt["Detected_Topic"] = topics
        ckpt["Summary"] = summaries
        ckpt["Key_Entities"] = [json.dumps(e) for e in entities_list]
        ckpt.to_csv(OUTPUT_CSV, index=False)

print("\nDone. Total time:", round((time.time() - start) / 60, 1), "minutes")

[10/2225] 0.31 art/s | elapsed 0.5 min | ETA 120.2 min
[20/2225] 0.31 art/s | elapsed 1.1 min | ETA 116.8 min
[30/2225] 0.28 art/s | elapsed 1.8 min | ETA 131.0 min
[40/2225] 0.28 art/s | elapsed 2.3 min | ETA 127.8 min
[50/2225] 0.27 art/s | elapsed 3.1 min | ETA 133.1 min
[60/2225] 0.28 art/s | elapsed 3.6 min | ETA 128.8 min
[70/2225] 0.28 art/s | elapsed 4.1 min | ETA 126.4 min
[80/2225] 0.29 art/s | elapsed 4.6 min | ETA 124.4 min
[90/2225] 0.28 art/s | elapsed 5.3 min | ETA 125.9 min
[100/2225] 0.28 art/s | elapsed 5.9 min | ETA 124.6 min
[110/2225] 0.29 art/s | elapsed 6.4 min | ETA 122.5 min
[120/2225] 0.29 art/s | elapsed 6.9 min | ETA 121.7 min
[130/2225] 0.29 art/s | elapsed 7.5 min | ETA 121.2 min
[140/2225] 0.29 art/s | elapsed 8.2 min | ETA 121.7 min
[150/2225] 0.28 art/s | elapsed 8.8 min | ETA 121.4 min
[160/2225] 0.29 art/s | elapsed 9.3 min | ETA 120.6 min
[170/2225] 0.29 art/s | elapsed 9.9 min | ETA 119.4 min
[180/2225] 0.29 art/s | elapsed 10.4 min | ETA 118.4 min


In [45]:
# Add the new columns to the main DataFrame
df["Detected_Topic"] = topics
df["Summary"] = summaries
df["Key_Entities"] = entities_list

# Final save (Key_Entities stored as JSON text)
save_df = df.copy()
save_df["Key_Entities"] = save_df["Key_Entities"].apply(json.dumps)
save_df.to_csv(OUTPUT_CSV, index=False)
print("Saved", len(df), "rows to", OUTPUT_CSV)

Saved 2225 rows to bbc_news_analysis_full.csv


### The new columns only

In [47]:
df[["Article_ID", "Detected_Topic", "Summary", "Key_Entities"]].head(10)

,Article_ID,Detected_Topic,Summary,Key_Entities
0,1,Business,TimeWarner's quarterly profits jumped 76% to $...,"[Richard Parsons, TimeWarner, Google, Warner B..."
1,2,Business,The dollar has reached its highest level again...,"[Alan Greenspan, Federal Reserve, Bank of Amer..."
2,3,Business,The owners of Yukos are asking the buyer of it...,"[Menatep Group, Rosneft, Yukos, Mikhail Khodor..."
3,4,Business,British Airways reported a 40% drop in profits...,"[Rod Eddington, Dresdner Kleinwort Wasserstein..."
4,5,Business,Shares in UK drinks and food firm Allied Domec...,"[Pernod Ricard, Allied Domecq, Wall Street Jou..."
5,6,Business,Japan's economy experienced a technical recess...,"[Heizo Takenaka, Lehman Brothers, Japan]"
6,7,Business,The US created fewer jobs than expected in Jan...,"[George W. Bush, Herbert Hoover, BMO Financial..."
7,8,Politics,"India's finance minister, Palaniappan Chidamba...","[Palaniappan Chidambaram, Gordon Brown, United..."
8,9,Business,Ethiopia's crop production increased by 24% in...,"[Henri Josserand, Food and Agriculture Organis..."
9,10,Politics,A US government claim accusing tobacco compani...,"[Clinton, Altria Group, RJ Reynolds Tobacco, L..."


### Final DataFrame: original + new columns together

In [48]:
df.head(30)

,Article_ID,Title,Article_Text,Original_Category,Detected_Topic,Summary,Key_Entities
0,1,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...,business,Business,TimeWarner's quarterly profits jumped 76% to $...,"[Richard Parsons, TimeWarner, Google, Warner B..."
1,2,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...,business,Business,The dollar has reached its highest level again...,"[Alan Greenspan, Federal Reserve, Bank of Amer..."
2,3,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...,business,Business,The owners of Yukos are asking the buyer of it...,"[Menatep Group, Rosneft, Yukos, Mikhail Khodor..."
3,4,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,business,Business,British Airways reported a 40% drop in profits...,"[Rod Eddington, Dresdner Kleinwort Wasserstein..."
4,5,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...,business,Business,Shares in UK drinks and food firm Allied Domec...,"[Pernod Ricard, Allied Domecq, Wall Street Jou..."
5,6,Japan narrowly escapes recession,Japan's economy teetered on the brink of a te...,business,Business,Japan's economy experienced a technical recess...,"[Heizo Takenaka, Lehman Brothers, Japan]"
6,7,Jobs growth still slow in the US,The US created fewer jobs than expected in Ja...,business,Business,The US created fewer jobs than expected in Jan...,"[George W. Bush, Herbert Hoover, BMO Financial..."
7,8,India calls for fair trade rules,"India, which attends the G7 meeting of seven ...",business,Politics,"India's finance minister, Palaniappan Chidamba...","[Palaniappan Chidambaram, Gordon Brown, United..."
8,9,Ethiopia's crop production up 24%,Ethiopia produced 14.27 million tonnes of cro...,business,Business,Ethiopia's crop production increased by 24% in...,"[Henri Josserand, Food and Agriculture Organis..."
9,10,Court rejects $280bn tobacco case,A US government claim accusing the country's ...,business,Politics,A US government claim accusing tobacco compani...,"[Clinton, Altria Group, RJ Reynolds Tobacco, L..."


### How often did the model agree with the dataset's own label?

In [49]:
agreement = (df["Detected_Topic"].str.lower() == df["Original_Category"].str.lower()).mean()
print(f"Model agreed with the original category on {agreement*100:.1f}% of {len(df)} articles.")

Model agreed with the original category on 81.7% of 2225 articles.


## Output as JSON

Export the merged DataFrame to JSON. Each record follows the structure
(`Article_ID`, `Title`, `Article_Text`, `Detected_Topic`, `Summary`, `Key_Entities`)

In [50]:
json_df = df.copy()
json_df["Article_Text"] = json_df["Article_Text"].str.strip().str.slice(0, 200) + "..."

records = json_df[
    ["Article_ID", "Title", "Article_Text", "Detected_Topic", "Summary", "Key_Entities"]
].to_dict(orient="records")

# show the first record
print(json.dumps(records[0], indent=2, ensure_ascii=False))

{
  "Article_ID": 1,
  "Title": "Ad sales boost Time Warner profit",
  "Article_Text": "Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, ...",
  "Detected_Topic": "Business",
  "Summary": "TimeWarner's quarterly profits jumped 76% to $1.13bn, driven by high-speed internet connections and advert sales. The company's film division saw profits slump 27% due to box-office flops. TimeWarner is projecting operating earnings growth of around 5% for 2005.",
  "Key_Entities": [
    "Richard Parsons",
    "TimeWarner",
    "Google",
    "Warner Bros",
    "AOL",
    "SEC",
    "Bertelsmann",
    "Lord of the Rings",
    "Alexander",
    "Catwoman"
  ]
}


In [51]:
with open("bbc_news_analysis_output.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print(f"Saved {len(records)} records to bbc_news_analysis_output.json")

Saved 2225 records to bbc_news_analysis_output.json
